In [8]:
import warnings

# Instalasi library yang diperlukan (-q untuk mode quiet agar output rapi)
!pip install pandas mlxtend openpyxl -q

print("✅ Library berhasil diinstal!")
warnings.filterwarnings("ignore")



✅ Library berhasil diinstal!


In [9]:
import pandas as pd

print("[TAHAP 1 & 2] Business & Data Understanding")

try:
    # 1. Baca file TANPA header terlebih dahulu untuk memindai isi file
    df_raw = pd.read_excel('data_mahasiswa.xlsx', sheet_name='Data Mahasiswa', header=None)

    # 2. Cari baris yang mengandung kata 'No' di kolom pertama (index 0)
    # .astype(str).str.strip() memastikan spasi tambahan tidak mengganggu pencarian
    header_idx = df_raw[df_raw[0].astype(str).str.strip() == 'No'].index[0]

    # 3. Baca ulang file dengan posisi header yang sudah ditemukan secara otomatis
    df = pd.read_excel('data_mahasiswa.xlsx', sheet_name='Data Mahasiswa', header=header_idx)

    # 4. Bersihkan data: Hapus baris 'TOTAL' atau baris kosong di bagian bawah
    # Kita hanya ambil baris yang kolom 'No'-nya berisi angka (data mahasiswa asli)
    df = df[df['No'].astype(str).str.isdigit()].reset_index(drop=True)

    print(f"✅ Data berhasil dimuat. Jumlah data mahasiswa valid: {len(df)} orang.")
    print("\n--- 5 Baris Pertama Data ---")
    display(df.head())

except FileNotFoundError:
    print("❌ Error: File 'data_mahasiswa.xlsx' tidak ditemukan.")
    print("   Pastikan nama file persis 'data_mahasiswa.xlsx' (huruf kecil semua) dan sudah diunggah di folder kiri.")
except IndexError:
    print("❌ Error: Kolom 'No' tidak ditemukan di file Excel.")
    print("   Pastikan sheet bernama 'Data Mahasiswa' dan memiliki kolom 'No'.")
except Exception as e:
    print(f"❌ Terjadi kesalahan yang tidak terduga: {e}")

[TAHAP 1 & 2] Business & Data Understanding
✅ Data berhasil dimuat. Jumlah data mahasiswa valid: 100 orang.

--- 5 Baris Pertama Data ---


,No,Nama,NIM,Data mining,Desain Grafis,Sistem Informasi Pendidikan,Teknologi IoT,Pengolahan Citra Digital,Pemrograman CMS,Realitas Virtual,Game Edukasi,Sistem Keamanan Jaringan
0,1,Ade,P03189001,✓,✗,✗,✓,✓,✓,✗,✗,✗
1,2,Arina,P03189002,✗,✗,✗,✓,✓,✓,✓,✗,✗
2,3,Dias,P03189003,✓,✗,✗,✗,✓,✓,✓,✗,✗
3,4,Bagas,P03189004,✗,✗,✗,✗,✓,✓,✓,✓,✓
4,5,Reyhan,P03189005,✓,✓,✗,✗,✓,✓,✗,✓,✗


In [10]:
print("[TAHAP 3] Data Preparation")

# Daftar kolom mata kuliah yang akan dianalisis polanya
kolom_mk = [
    'Data mining', 'Desain Grafis', 'Sistem Informasi Pendidikan',
    'Teknologi IoT', 'Pengolahan Citra Digital', 'Pemrograman CMS',
    'Realitas Virtual', 'Game Edukasi', 'Sistem Keamanan Jaringan'
]

# Ambil hanya kolom mata kuliah (buang No, Nama, NIM untuk proses mining)
df_mk = df[kolom_mk].copy()

# Ubah '✓' menjadi True dan '✗' menjadi False
df_mk_bool = df_mk.replace({'✓': True, '✗': False})

print("✅ Data berhasil diubah ke format Boolean (True/False).")
print("\n--- Contoh Data Setelah Preparasi ---")
display(df_mk_bool.head())

[TAHAP 3] Data Preparation
✅ Data berhasil diubah ke format Boolean (True/False).

--- Contoh Data Setelah Preparasi ---


,Data mining,Desain Grafis,Sistem Informasi Pendidikan,Teknologi IoT,Pengolahan Citra Digital,Pemrograman CMS,Realitas Virtual,Game Edukasi,Sistem Keamanan Jaringan
0,True,False,False,True,True,True,False,False,False
1,False,False,False,True,True,True,True,False,False
2,True,False,False,False,True,True,True,False,False
3,False,False,False,False,True,True,True,True,True
4,True,True,False,False,True,True,False,True,False


In [11]:
from mlxtend.frequent_patterns import apriori, association_rules

print("[TAHAP 4] Modeling")

# A. Mencari Frequent Itemsets (Pola yang sering muncul)
# min_support=0.10 berarti pola harus muncul di minimal 10% mahasiswa (10 dari 100 orang)
# Jika hasil terlalu sedikit, turunkan jadi 0.05. Jika terlalu banyak, naikkan jadi 0.15
frequent_itemsets = apriori(df_mk_bool, min_support=0.10, use_colnames=True)
print(f"✅ Ditemukan {len(frequent_itemsets)} kombinasi mata kuliah yang sering muncul.")

# B. Membuat Aturan Asosiasi (Association Rules)
# min_threshold=0.5 berarti kita hanya ambil aturan dengan keyakinan (confidence) minimal 50%
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)
print(f"✅ Ditemukan {len(rules)} aturan asosiasi yang valid.")

[TAHAP 4] Modeling
✅ Ditemukan 105 kombinasi mata kuliah yang sering muncul.
✅ Ditemukan 380 aturan asosiasi yang valid.


In [12]:
print("[TAHAP 5] Evaluation")

# Urutkan aturan berdasarkan nilai 'lift' dari terbesar ke terkecil
rules_sorted = rules.sort_values(by='lift', ascending=False).reset_index(drop=True)

# Tampilkan 10 aturan teratas
kolom_tampilan = ['antecedents', 'consequents', 'support', 'confidence', 'lift']
print("\n--- 10 Aturan Asosiasi Terbaik (Lift Tertinggi) ---")
display(rules_sorted[kolom_tampilan].head(10))

print("\n💡 PANDUAN MEMBACA HASIL UNTUK SKRIPSI:")
print("• antecedents : Mata kuliah yang diambil lebih dulu (JIKA).")
print("• consequents : Mata kuliah yang direkomendasikan (MAKA).")
print("• support     : Seberapa sering kombinasi ini muncul di seluruh data (misal 0.15 = 15%).")
print("• confidence  : Seberapa yakin (%) consequents akan diambil jika antecedents diambil.")
print("• lift        : Kekuatan hubungan. Lift > 1 berarti hubungan positif (saling menguatkan).")

[TAHAP 5] Evaluation

--- 10 Aturan Asosiasi Terbaik (Lift Tertinggi) ---


,antecedents,consequents,support,confidence,lift
0,"(Data mining, Game Edukasi, Pengolahan Citra D...","(Pemrograman CMS, Teknologi IoT)",0.10,0.526316,1.461988
1,"(Realitas Virtual, Game Edukasi, Teknologi IoT...",(Data mining),0.10,0.909091,1.420455
2,"(Game Edukasi, Pengolahan Citra Digital, Pemro...","(Data mining, Realitas Virtual)",0.10,0.769231,1.398601
3,"(Game Edukasi, Pemrograman CMS, Teknologi IoT)","(Data mining, Pengolahan Citra Digital, Realit...",0.10,0.500000,1.388889
4,"(Data mining, Game Edukasi, Pengolahan Citra D...",(Teknologi IoT),0.10,0.526316,1.385042
5,"(Realitas Virtual, Data mining, Game Edukasi, ...",(Teknologi IoT),0.10,0.526316,1.385042
6,"(Desain Grafis, Data mining)","(Game Edukasi, Pemrograman CMS)",0.13,0.812500,1.354167
7,"(Pemrograman CMS, Sistem Keamanan Jaringan)","(Game Edukasi, Realitas Virtual)",0.15,0.714286,1.347709
8,"(Game Edukasi, Pengolahan Citra Digital, Tekno...","(Data mining, Pemrograman CMS, Realitas Virtual)",0.10,0.714286,1.322751
9,"(Game Edukasi, Pengolahan Citra Digital, Reali...","(Data mining, Pemrograman CMS)",0.10,0.833333,1.322751



💡 PANDUAN MEMBACA HASIL UNTUK SKRIPSI:
• antecedents : Mata kuliah yang diambil lebih dulu (JIKA).
• consequents : Mata kuliah yang direkomendasikan (MAKA).
• support     : Seberapa sering kombinasi ini muncul di seluruh data (misal 0.15 = 15%).
• confidence  : Seberapa yakin (%) consequents akan diambil jika antecedents diambil.
• lift        : Kekuatan hubungan. Lift > 1 berarti hubungan positif (saling menguatkan).


In [13]:
print("[TAHAP 6] Deployment")

# 1. Simpan hasil aturan ke file Excel baru
output_filename = 'hasil_aturan_asosiasi.xlsx'
rules_sorted.to_excel(output_filename, index=False)
print(f"✅ Hasil aturan asosiasi berhasil disimpan sebagai '{output_filename}'.")
print("   (Klik ikon Folder di kiri, cari file tersebut, klik titik tiga, lalu Download)")

# 2. Fungsi Decision Support System (Rekomendasi Sederhana)
def beri_rekomendasi(mata_kuliah_diminati):
    # Cari aturan di mana mata kuliah yang diminati ada di sisi kiri (antecedents)
    aturan_terkait = rules_sorted[rules_sorted['antecedents'].apply(lambda x: mata_kuliah_diminati in x)]

    if aturan_terkait.empty:
        return f"⚠️ Tidak ditemukan pola asosiasi yang kuat untuk '{mata_kuliah_diminati}' dengan threshold saat ini."

    # Ambil rekomendasi dengan confidence tertinggi (baris pertama setelah di-sort)
    rek_best = aturan_terkait.iloc[0]
    mk_rekomendasi = list(rek_best['consequents'])
    confidence = rek_best['confidence'] * 100

    return f"💡 Jika mahasiswa mengambil '{mata_kuliah_diminati}', sistem merekomendasikan juga mengambil: {', '.join(mk_rekomendasi)} (Keyakinan/Confidence: {confidence:.1f}%)"

print("\n--- CONTOH PENERAPAN DECISION SUPPORT SYSTEM ---")
print(beri_rekomendasi('Data mining'))
print(beri_rekomendasi('Pengolahan Citra Digital'))
print(beri_rekomendasi('Desain Grafis'))

[TAHAP 6] Deployment
✅ Hasil aturan asosiasi berhasil disimpan sebagai 'hasil_aturan_asosiasi.xlsx'.
   (Klik ikon Folder di kiri, cari file tersebut, klik titik tiga, lalu Download)

--- CONTOH PENERAPAN DECISION SUPPORT SYSTEM ---
💡 Jika mahasiswa mengambil 'Data mining', sistem merekomendasikan juga mengambil: Pemrograman CMS, Teknologi IoT (Keyakinan/Confidence: 52.6%)
💡 Jika mahasiswa mengambil 'Pengolahan Citra Digital', sistem merekomendasikan juga mengambil: Pemrograman CMS, Teknologi IoT (Keyakinan/Confidence: 52.6%)
💡 Jika mahasiswa mengambil 'Desain Grafis', sistem merekomendasikan juga mengambil: Game Edukasi, Pemrograman CMS (Keyakinan/Confidence: 81.2%)


In [15]:
!pip install streamlit pandas mlxtend openpyxl -q